In [6]:
import pandas as pd

In [37]:
url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-10.parquet'
columns = ['lpep_pickup_datetime', 'lpep_dropoff_datetime', 'PULocationID', 'DOLocationID', 'passenger_count', 'trip_distance', 'tip_amount', 'total_amount']
df = pd.read_parquet(url, columns=columns)
df = df.fillna(0)

# THE FIX: Force the exact string format Flink expects (strips microseconds!)
df['lpep_pickup_datetime'] = df['lpep_pickup_datetime'].dt.strftime('%Y-%m-%d %H:%M:%S')
df['lpep_dropoff_datetime'] = df['lpep_dropoff_datetime'].dt.strftime('%Y-%m-%d %H:%M:%S')

# Keep the chronological sort to keep the Watermark happy
df = df.sort_values('lpep_pickup_datetime')


In [38]:
df.head()

,lpep_pickup_datetime,lpep_dropoff_datetime,PULocationID,DOLocationID,passenger_count,trip_distance,tip_amount,total_amount
86,2025-09-24 22:46:18,2025-09-24 23:38:39,138,265,2.0,25.06,0.00,206.00
166,2025-09-24 23:40:05,2025-09-24 23:40:13,265,265,5.0,0.00,0.00,51.00
4547,2025-09-27 23:47:44,2025-09-27 23:57:53,92,135,2.0,2.34,4.20,25.20
4624,2025-09-28 01:11:55,2025-09-28 01:26:20,82,95,1.0,2.46,3.42,20.52
4670,2025-09-28 02:24:54,2025-09-28 02:31:48,92,138,1.0,3.01,4.14,24.84


In [39]:
from models import Ride, ride_from_row, ride_serializer

In [40]:
from kafka import KafkaProducer

server = 'localhost:9092'

producer = KafkaProducer(   
    bootstrap_servers=[server],
    value_serializer=ride_serializer
)

In [41]:
import dataclasses


topic_name = 'green-trips'


In [42]:
df=df.fillna(0)

In [43]:
import time

t0 = time.time()

for _, row in df.iterrows():
    ride = ride_from_row(row)
    producer.send(topic_name, value=ride)

producer.flush()

t1 = time.time()
print(f'took {(t1 - t0):.2f} seconds')

took 27.69 seconds
